---
# HazardNet Event-Based Unified Baseline Comparison
---
### PURPOSE:
  Train and evaluate 2 baseline architectures (`3D ResNet-18` and `TimeSformer` against HazardNet using identical
  data, loss functions, and evaluation metrics for fair  comparison.

---


In [1]:
"""
================================================================================
HazardNet Baseline Comparison: 3D ResNet-18 + TimeSformer (Q1 Journal Edition)
================================================================================
EDGE-FIRST & GREEN AI ALIGNED:
  • Automatic Mixed Precision (AMP) for VRAM safety & energy efficiency
  • Inference Latency & Model Size Benchmarking (Edge-First Autonomy mandate)
  • Bangladesh-Calibrated Severity Normalization (Hybrid Cognitive Architecture)
  • Macro F1-Score tracking for imbalanced multi-class environmental hazards
  • Strict VRAM Garbage Collection to prevent Kaggle OOM across 5 folds
================================================================================
"""
import os, json, time, gc, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, h5py, matplotlib
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from tqdm import tqdm

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# BANGLADESH SEVERITY THRESHOLDS & NORMALIZER (Physical Grounding)
# ============================================================================
HAZARD_TYPES = ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']

SEVERITY_THRESHOLDS = {
    "Cold Wave": {"anchors": [(16, 0.10), (13, 0.30), (10, 0.50), (8, 0.70), (6, 0.90), (4, 1.0)]},
    "Heat Wave": {"anchors": [(36, 0.25), (38, 0.50), (40, 0.70), (42, 0.85), (44, 1.0)]},
    "Flood": {"anchors": [(-0.5, 0.30), (0.0, 0.50), (1.0, 0.75), (2.0, 1.0)]},
    "Flash Flood": {"anchors": [(44, 0.40), (88, 0.65), (150, 0.85), (250, 1.0)]},
    "Drought": {"anchors": [(-1.0, 0.30), (-1.5, 0.60), (-2.0, 0.85), (-2.5, 1.0)]},
    "Fire": {"anchors": [(11.2, 0.30), (21.3, 0.55), (38.0, 0.75), (50.0, 0.90), (70.0, 1.0)]},
    "Severe Local Storm": {"anchors": [(45, 0.25), (61, 0.40), (91, 0.65), (121, 0.90), (150, 1.0)]},
    "Tropical Cyclone": {"anchors": [(63, 0.25), (89, 0.50), (118, 0.70), (166, 0.85), (221, 1.0)]},
}

class SeverityNormalizer:
    def __init__(self, hazard: str):
        cfg = SEVERITY_THRESHOLDS[hazard]
        xs = np.array([a[0] for a in cfg["anchors"]], dtype=float)
        ys = np.array([a[1] for a in cfg["anchors"]], dtype=float)
        self._flip = xs[0] > xs[-1]
        if self._flip: xs = -xs
        self.xs, self.ys = xs, ys

    def _to_internal(self, x): return -x if self._flip else x
    def to_severity(self, x: float) -> float:
        x = self._to_internal(float(x))
        return float(np.interp(x, self.xs, self.ys, left=self.ys[0], right=self.ys[-1]))

# ============================================================================
# CONFIGURATION
# ============================================================================
class TrainConfig:
    EXPERIMENTAL_DIR = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/HazardNet_Event_Based_Datasets/event_kfold'
    MASTER_H5_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5'
    CONFIG_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json'
    OUTPUT_DIR = '/kaggle/working/HazardNet_Baseline_Results_Heavy'
    BATCH_SIZE = 6  # Strictly reduced for TimeSformer/ResNet3D VRAM safety on 16GB GPUs
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4 # Lower LR for fine-tuning massive pretrained backbones
    WEIGHT_DECAY = 1e-4
    PATIENCE = 10
    GRAD_CLIP = 1.0
    NUM_WORKERS = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(TrainConfig.OUTPUT_DIR, exist_ok=True)
torch.backends.cudnn.benchmark = True # Accelerate 3D convolutions

# ============================================================================
# BASELINE ARCHITECTURES
# ============================================================================
class ResNet3D(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        from torchvision.models.video import r3d_18, R3D_18_Weights
        self.backbone = r3d_18(weights=R3D_18_Weights.DEFAULT)
        old_conv = self.backbone.stem[0]
        self.backbone.stem[0] = nn.Conv3d(in_channels, old_conv.out_channels, kernel_size=old_conv.kernel_size, stride=old_conv.stride, padding=old_conv.padding, bias=False)
        self.backbone.fc = nn.Identity()
        self.feature_dim = 512
        self.shared_fc = nn.Sequential(nn.Linear(self.feature_dim, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.backbone(x)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)

class TimeSformerBaseline(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        from transformers import TimesformerForVideoClassification, TimesformerConfig
        try:
            self.backbone = TimesformerForVideoClassification.from_pretrained(
                "facebook/timesformer-base-finetuned-k400", num_labels=num_hazards, ignore_mismatched_sizes=True)
        except Exception:
            config = TimesformerConfig(num_channels=in_channels, num_frames=10, image_size=64, patch_size=16, 
                                       hidden_size=768, num_hidden_layers=12, num_attention_heads=12, num_labels=num_hazards)
            self.backbone = TimesformerForVideoClassification(config)

        self.backbone.config.num_channels = in_channels
        self.backbone.config.image_size = 64
        self.backbone.config.patch_size = 16
        self.backbone.config.num_frames = 10

        # Robust Patch Embedding Replacement (Handles HF API variations)
        try:
            old_proj = self.backbone.timesformer.embeddings.patch_embeddings.projection
            if isinstance(old_proj, nn.Conv3d):
                self.backbone.timesformer.embeddings.patch_embeddings.projection = nn.Conv3d(
                    in_channels, old_proj.out_channels, kernel_size=old_proj.kernel_size, stride=old_proj.stride, padding=old_proj.padding, bias=(old_proj.bias is not None))
            else:
                self.backbone.timesformer.embeddings.patch_embeddings.projection = nn.Conv2d(
                    in_channels, old_proj.out_channels, kernel_size=old_proj.kernel_size, stride=old_proj.stride, padding=old_proj.padding, bias=(old_proj.bias is not None))
        except AttributeError:
            # Fallback for newer transformers versions
            out_c = 768
            self.backbone.timesformer.embeddings.patch_embeddings.projection = nn.Conv3d(
                in_channels, out_c, kernel_size=(1, 16, 16), stride=(1, 16, 16), padding=0)

        self.backbone.timesformer.embeddings.image_size = 64
        self.backbone.timesformer.embeddings.patch_size = 16
        self.backbone.timesformer.embeddings.num_channels = in_channels
        self.backbone.timesformer.embeddings.num_frames = 10
        self.backbone.timesformer.embeddings.num_patches = (64 // 16) ** 2
        
        self.backbone.classifier = nn.Identity()
        self.feature_dim = 768
        self.shared_fc = nn.Sequential(nn.Linear(self.feature_dim, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        pixel_values = x.permute(0, 2, 1, 3, 4) # (B, C, T, H, W) -> (B, T, C, H, W)
        outputs = self.backbone(pixel_values=pixel_values, output_hidden_states=True)
        features = outputs.hidden_states[-1][:, 0] # CLS Token
        features = self.shared_fc(features)
        return self.hazard_head(features), self.severity_head(features).squeeze(1)

def get_model(model_name, in_channels=15, num_hazards=8):
    if model_name == 'resnet3d': return ResNet3D(in_channels, num_hazards)
    elif model_name == 'timesformer': return TimeSformerBaseline(in_channels, num_hazards)
    else: raise ValueError(f"Unknown model: {model_name}")

# ============================================================================
# LOSS, DATASET & METRICS (Identical to HazardNet v3.0)
# ============================================================================
class HomoscedasticMTLLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')
        self.huber_loss = nn.SmoothL1Loss(reduction='none')

    def forward(self, hazard_pred, severity_pred, hazard_true, severity_true, confidence):
        loss_cls = self.ce_loss(hazard_pred, hazard_true)
        loss_reg = self.huber_loss(severity_pred, severity_true)
        prec_cls, prec_reg = torch.exp(-self.log_vars[0]), torch.exp(-self.log_vars[1])
        total = (prec_cls * (loss_cls * confidence).mean() + self.log_vars[0]) + \
                (prec_reg * (loss_reg * confidence).mean() + self.log_vars[1])
        return total, (loss_cls * confidence).mean().item(), (loss_reg * confidence).mean().item()

class MasterHDF5Dataset(Dataset):
    def __init__(self, csv_path, master_h5_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.master_h5_path = master_h5_path
        self.augment = augment
        self.h5f = None
        self._worker_id = None
        self.brightness, self.contrast, self.temporal_shift = 0.1, 0.1, 1
        self.target_shape = (15, 10, 64, 64)
        self._normalizers = {h: SeverityNormalizer(h) for h in HAZARD_TYPES}

    def _open_h5(self):
        wid = get_worker_info().id if get_worker_info() else -1
        if self.h5f is None or self._worker_id != wid:
            if self.h5f: self.h5f.close()
            self.h5f = h5py.File(self.master_h5_path, 'r', rdcc_nbytes=1024**2*10)
            self._worker_id = wid

    def __len__(self): return len(self.df)

    def _resize_spatial(self, tensor):
        c, t, h, w = tensor.shape
        th, tw = self.target_shape[2], self.target_shape[3]
        if h == th and w == tw: return tensor
        r = tensor.permute(1,0,2,3).reshape(t*c,1,h,w)
        r = F.interpolate(r, size=(th,tw), mode='nearest')
        return r.reshape(t,c,th,tw).permute(1,0,2,3).contiguous()

    # FIX: Restored missing augmentation method to ensure fair comparison
    def _augment(self, tensor):
        if np.random.rand() > 0.5:
            tensor = tensor + np.random.uniform(-self.brightness, self.brightness)
        if np.random.rand() > 0.5:
            f = 1.0 + np.random.uniform(-self.contrast, self.contrast)
            m = tensor.mean(dim=[-1,-2], keepdim=True)
            tensor = (tensor - m) * f + m
        if np.random.rand() > 0.5:
            s = np.random.randint(-self.temporal_shift, self.temporal_shift+1)
            if s > 0:
                b = tensor[:,0:1,:,:].repeat(1,s,1,1)
                tensor = torch.cat([b, tensor[:,:-s,:,:]], dim=1)
            elif s < 0:
                a = abs(s); b = tensor[:,-1:,:,:].repeat(1,a,1,1)
                tensor = torch.cat([tensor[:,a:,:,:], b], dim=1)
        return tensor

    def __getitem__(self, idx):
        self._open_h5()
        row = self.df.iloc[idx]
        eid = str(row['event_id'])
        tensor = torch.from_numpy(self.h5f['tensors'][eid][:]).float()
        label = int(row['hazard_idx'])
        hazard = HAZARD_TYPES[label]
        severity = float(row.get('severity_index', 0.0))
        src = row.get('severity_source_index', None)
        if src is not None and not pd.isna(src): 
            severity = self._normalizers[hazard].to_severity(float(src))
        confidence = float(row.get('confidence', 0.5))
        tensor = self._resize_spatial(tensor)
        if self.augment: tensor = self._augment(tensor) # FIX: Applied augmentation
        return tensor, label, severity, confidence, eid

class EnhancedMetricsTracker:
    def __init__(self): self.reset()
    def reset(self):
        self.total_losses, self.hazard_preds, self.hazard_targets = [], [], []
        self.severity_preds, self.severity_targets = [], []
    def update(self, total_loss, h_pred, h_true, s_pred, s_true):
        self.total_losses.append(total_loss)
        self.hazard_preds.extend(h_pred); self.hazard_targets.extend(h_true)
        self.severity_preds.extend(s_pred); self.severity_targets.extend(s_true)
    def get_summary(self):
        h_acc = accuracy_score(self.hazard_targets, self.hazard_preds)
        h_f1_macro = f1_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0)
        s_mse = mean_squared_error(self.severity_targets, self.severity_preds) if self.severity_targets else 0.0
        r2 = r2_score(self.severity_targets, self.severity_preds) if len(set(self.severity_targets)) > 1 else 0.0
        return {'loss_total': np.mean(self.total_losses), 'hazard_accuracy': h_acc, 'hazard_f1_macro': h_f1_macro,
                'severity_rmse': np.sqrt(s_mse), 'severity_r2': r2}

# ============================================================================
# EDGE-FIRST BENCHMARKING (Latency & Size)
# ============================================================================
def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024 / 1024

def benchmark_latency_ms(model, device, batch_size=1, num_iterations=50):
    model.eval()
    dummy_input = torch.randn(batch_size, 15, 10, 64, 64, device=device)
    for _ in range(10): 
        with torch.no_grad(), autocast(): _ = model(dummy_input)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad(), autocast():
        for _ in range(num_iterations): _ = model(dummy_input)
    if device.type == 'cuda': torch.cuda.synchronize()
    return (time.time() - start) / num_iterations * 1000

# ============================================================================
# TRAINING LOOP (With AMP for Green AI / Energy Efficiency)
# ============================================================================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train(); metrics = EnhancedMetricsTracker()
    for tensors, cls_idx, severity, confidence, _ in tqdm(loader, desc="Train", unit="batch"):
        tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
        optimizer.zero_grad()
        with autocast():
            h_pred, s_pred = model(tensors)
            total, _, _ = criterion(h_pred, s_pred, cls_idx, severity, confidence)
        scaler.scale(total).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=TrainConfig.GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        metrics.update(total.item(), h_pred.detach().argmax(1).cpu().numpy(), cls_idx.cpu().numpy(), s_pred.detach().cpu().numpy(), severity.cpu().numpy())
    return metrics

def evaluate(model, loader, criterion, device):
    model.eval(); metrics = EnhancedMetricsTracker()
    with torch.no_grad():
        for tensors, cls_idx, severity, confidence, _ in tqdm(loader, desc="Val", unit="batch"):
            tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
            with autocast():
                h_pred, s_pred = model(tensors)
                total, _, _ = criterion(h_pred, s_pred, cls_idx, severity, confidence)
            metrics.update(total.item(), h_pred.argmax(1).cpu().numpy(), cls_idx.cpu().numpy(), s_pred.cpu().numpy(), severity.cpu().numpy())
    return metrics

def train_baseline(model_name, fold_idx, num_classes, output_dir):
    safe_name = f"{model_name}_fold{fold_idx}"
    print(f"\n{'─'*60}\n📂 {safe_name}\n{'─'*60}")
    fold_dir = os.path.join(TrainConfig.EXPERIMENTAL_DIR, f'fold_{fold_idx}')
    train_loader = DataLoader(MasterHDF5Dataset(os.path.join(fold_dir, 'train_events.csv'), TrainConfig.MASTER_H5_PATH, True), batch_size=TrainConfig.BATCH_SIZE, shuffle=True, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(MasterHDF5Dataset(os.path.join(fold_dir, 'val_events.csv'), TrainConfig.MASTER_H5_PATH, False), batch_size=TrainConfig.BATCH_SIZE, shuffle=False, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(MasterHDF5Dataset(os.path.join(fold_dir, 'test_events.csv'), TrainConfig.MASTER_H5_PATH, False), batch_size=TrainConfig.BATCH_SIZE, shuffle=False, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)

    model = get_model(model_name, 15, num_classes).to(TrainConfig.DEVICE)
    criterion = HomoscedasticMTLLoss().to(TrainConfig.DEVICE)
    optimizer = AdamW([{'params': model.parameters()}, {'params': criterion.log_vars}], lr=TrainConfig.LEARNING_RATE, weight_decay=TrainConfig.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=TrainConfig.NUM_EPOCHS, eta_min=1e-6)
    scaler = GradScaler()

    best_val_loss, patience_counter = float('inf'), 0
    ckpt_path = os.path.join(output_dir, f'{safe_name}_best.pt')

    for epoch in range(TrainConfig.NUM_EPOCHS):
        train_m = train_epoch(model, train_loader, optimizer, criterion, TrainConfig.DEVICE, scaler)
        val_m = evaluate(model, val_loader, criterion, TrainConfig.DEVICE)
        scheduler.step()
        ts, vs = train_m.get_summary(), val_m.get_summary()
        if vs['loss_total'] < best_val_loss:
            best_val_loss = vs['loss_total']; patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= TrainConfig.PATIENCE: print(f"  ⏹️ Early stopping at epoch {epoch+1}"); break
        if (epoch+1) % 10 == 0: print(f"  Epoch {epoch+1:2d} | Train Acc:{ts['hazard_accuracy']:.3f} | Val Acc:{vs['hazard_accuracy']:.3f}")

    model.load_state_dict(torch.load(ckpt_path))
    test_metrics = evaluate(model, test_loader, criterion, TrainConfig.DEVICE)
    s = test_metrics.get_summary()
    
    # Edge-First Benchmarking
    size_mb = get_model_size_mb(model)
    latency_ms = benchmark_latency_ms(model, TrainConfig.DEVICE, batch_size=1)
    
    print(f"  🏆 Test: Acc={s['hazard_accuracy']:.4f} MacroF1={s['hazard_f1_macro']:.4f} RMSE={s['severity_rmse']:.4f} R²={s['severity_r2']:.4f}")
    print(f"  ⚡ Edge Metrics: Size={size_mb:.2f} MB | Latency={latency_ms:.2f} ms/sample")
    
    # FIX: Strict VRAM Cleanup to prevent Kaggle OOM across sequential folds
    del model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    torch.cuda.empty_cache()
    gc.collect()
    
    return {'model': model_name, 'fold': fold_idx, 'size_mb': size_mb, 'latency_ms': latency_ms,
            'accuracy': s['hazard_accuracy'], 'f1_macro': s['hazard_f1_macro'], 'rmse': s['severity_rmse'], 'r2': s['severity_r2']}

# ============================================================================
# MAIN ORCHESTRATOR
# ============================================================================
MODEL_MAP = {'resnet3d': '3D ResNet-18 (Heavyweight 3D CNN)', 'timesformer': 'TimeSformer (Divided Space-Time Transformer)'}
MODEL = 'all'  # Options: 'resnet3d', 'timesformer', 'all'

def main():
    print("=" * 80, "\n🏗️ HAZARDNET HEAVYWEIGHT BASELINE TRAINING (Edge-First & Green AI Aligned)\n" + "=" * 80)
    with open(TrainConfig.CONFIG_PATH, 'r') as f: config = json.load(f)
    num_classes = config['n_classes']
    models = list(MODEL_MAP.keys()) if MODEL == 'all' else [MODEL]
    all_results = {m: [] for m in models}

    for model_name in models:
        print(f"\n🚀 BASELINE: {MODEL_MAP[model_name].upper()}")
        model_output = os.path.join(TrainConfig.OUTPUT_DIR, model_name)
        os.makedirs(model_output, exist_ok=True)
        for fold_idx in range(5):
            result = train_baseline(model_name, fold_idx, num_classes, model_output)
            all_results[model_name].append(result)
        pd.DataFrame(all_results[model_name]).to_csv(os.path.join(model_output, f'{model_name}_results.csv'), index=False)

    print(f"\n{'='*80}\n📊 HEAVYWEIGHT vs EDGE-FIRST COMPARISON TABLE (IEEE TGRS)\n{'='*80}")
    comparison_rows = []
    for m_name, m_label in MODEL_MAP.items():
        res = all_results.get(m_name, [])
        if res:
            comparison_rows.append({
                'Model': m_label, 
                'Size (MB)': f"{np.mean([r['size_mb'] for r in res]):.2f}",
                'Latency (ms)': f"{np.mean([r['latency_ms'] for r in res]):.2f}",
                'Accuracy': f"{np.mean([r['accuracy'] for r in res]):.4f} ± {np.std([r['accuracy'] for r in res]):.4f}",
                'Macro F1': f"{np.mean([r['f1_macro'] for r in res]):.4f} ± {np.std([r['f1_macro'] for r in res]):.4f}",
                'RMSE': f"{np.mean([r['rmse'] for r in res]):.4f} ± {np.std([r['rmse'] for r in res]):.4f}",
            })
            
    # Add HazardNet CNN Reference Metrics
    comparison_rows.append({
        'Model': 'HazardNet 3D-CNN (Ours)', 
        'Size (MB)': '~4.20', 'Latency (ms)': '~18.50',
        'Accuracy': '0.9887 ± 0.0038', 'Macro F1': '0.9887 ± 0.0038', 'RMSE': '0.1238 ± 0.0069'
    })

    df_comp = pd.DataFrame(comparison_rows)
    print(df_comp.to_string(index=False))
    df_comp.to_csv(os.path.join(TrainConfig.OUTPUT_DIR, 'heavyweight_baseline_comparison.csv'), index=False)
    print(f"\n✅ All heavyweight baseline training complete! Results: {TrainConfig.OUTPUT_DIR}")

if __name__ == '__main__':
    main()

🏗️ HAZARDNET HEAVYWEIGHT BASELINE TRAINING (Edge-First & Green AI Aligned)

🚀 BASELINE: 3D RESNET-18 (HEAVYWEIGHT 3D CNN)

────────────────────────────────────────────────────────────
📂 resnet3d_fold0
────────────────────────────────────────────────────────────
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 192MB/s]
/tmp/ipykernel_22/3230835593.py:310: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292

  Epoch 10 | Train Acc:0.973 | Val Acc:0.947


Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWa

  Epoch 20 | Train Acc:0.992 | Val Acc:0.978


...

  Epoch 30 | Train Acc:0.997 | Val Acc:0.971


Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/69 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 0/322 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:277: FutureWa

  ⏹️ Early stopping at epoch 40


Val:   0%|          | 0/98 [00:00<?, ?batch/s]/tmp/ipykernel_22/3230835593.py:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:09<00:00, 10.17batch/s]
/tmp/ipykernel_22/3230835593.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(): _ = model(dummy_input)
/tmp/ipykernel_22/3230835593.py:264: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


  🏆 Test: Acc=0.9590 MacroF1=0.9499 RMSE=0.1399 R²=0.8456
  ⚡ Edge Metrics: Size=471.98 MB | Latency=19.53 ms/sample

📊 HEAVYWEIGHT vs EDGE-FIRST COMPARISON TABLE (IEEE TGRS)
                                       Model Size (MB) Latency (ms)        Accuracy        Macro F1            RMSE
           3D ResNet-18 (Heavyweight 3D CNN)    127.27         3.46 0.9942 ± 0.0017 0.9899 ± 0.0079 0.1100 ± 0.0073
TimeSformer (Divided Space-Time Transformer)    471.98        19.21 0.9666 ± 0.0061 0.9653 ± 0.0104 0.1401 ± 0.0099
                     HazardNet 3D-CNN (Ours)     ~4.20       ~18.50 0.9887 ± 0.0038 0.9887 ± 0.0038 0.1238 ± 0.0069

✅ All heavyweight baseline training complete! Results: /kaggle/working/HazardNet_Baseline_Results_Heavy
